## IsTheWorldReadyForTheNextPandemic - Mini Project
### Project handled by Liza, Ravit, Hagit and Hodaya

**This notebook includes assembling the feature matrix of the model**



---
## Part 3 · Assembling the Featue Matrix


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Core imports.
import numpy as np
np.random.seed(42)  # For reproducibility
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set the path where your DataSet CSVs located.
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'


## Assembling the feature Matrix:
1. X1 - מיטות אשפוז לכל 1,000 תושבים (%)
2. x2 - רופאים לכל 1,000 תושבים (%)
3. X3 - ציון "Early Detection & Reporting" ממדד ה-GHS
4. X4 - שיעור בדיקות קורונה בשנת 2021 ל-1,000 איש (%)
5. X5 - ציון מדד GHSI הכללי
6. X6 - אחוז משתמשי האינטרנט במדינה (%)
7. X7 - אחוז בני +60 באוכלוסייה (%)
8. X8 - שכיחות המעשנים באוכלוסיה (%)
9. X9 - שכיחות הסוכרתיים באוכלוסיה (%)

In [ ]:
import os
import pandas as pd

FOLDER_PATH = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

# 1. בדיקת עמודות קובץ הבריאות
print("--- עמודות בקובץ הבריאות (מיזוג 1) ---")
try:
    df_health = pd.read_csv(os.path.join(FOLDER_PATH, 'cleaned_health_demographics_final.csv'))
    print(df_health.columns.tolist()[:15]) # 15 הראשונות
except Exception as e:
    print(f"שגיאה בטעינת קובץ בריאות: {e}")

# 2. בדיקת עמודות קובץ האפידמיולוגיה (הקובץ שתיקנו עכשיו)
print("\n--- עמודות בקובץ האפידמיולוגיה (מיזוג 4) ---")
try:
    df_epi = pd.read_csv(os.path.join(FOLDER_PATH, 'merged_epidemiology_demographics_yearly.csv'))
    print(df_epi.columns.tolist()[:15])
except Exception as e:
    print(f"שגיאה בטעינת קובץ אפידמיולוגיה: {e}")

# 3. בדיקת עמודות קובץ ה-GHS והטכנולוגיה שבזיכרון
print("\n--- עמודות במשתנה הזיכרון df_ict_ghs_cleaned (מיזוג 3) ---")
try:
    print(df_ict_ghs_cleaned.columns.tolist()[:15])
except Exception as e:
    print(f"משתנה df_ict_ghs_cleaned לא נמצא בזיכרון! שגיאה: {e}")


--- עמודות בקובץ הבריאות (מיזוג 1) ---
['location_key', 'country_code', 'country_name', 'aggregation_level', 'population', 'population_male', 'population_female', 'population_rural', 'population_urban', 'population_largest_city', 'population_clustered', 'population_density', 'human_development_index', 'population_age_00_09', 'population_age_10_19']

--- עמודות בקובץ האפידמיולוגיה (מיזוג 4) ---
['location_key', 'YEAR', 'country_code', 'country_name', 'aggregation_level', 'new_confirmed', 'new_deceased', 'new_recovered', 'new_tested', 'cumulative_confirmed', 'cumulative_deceased', 'cumulative_recovered', 'cumulative_tested', 'population', 'population_male']

--- עמודות במשתנה הזיכרון df_ict_ghs_cleaned (מיזוג 3) ---
['Entity', 'Code', 'Year', 'Landline phone subscriptions', 'Landline Internet subscriptions', 'Mobile phone subscriptions', 'Internet users', 'location_key', 'country_code', 'country_name', 'aggregation_level', 'Country', 'OVERALL SCORE', '1) PREVENTION OF THE EMERGENCE OR RE

In [ ]:
import os
import pandas as pd
import numpy as np

# הגדרת נתיב העבודה שלכן (ליזה, רווית, חגית והודיה)
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

print("1. טוען את קבצי המקור המעודכנים...")
df_master_epi_demo = pd.read_csv(os.path.join(data_dir, 'merged_epidemiology_demographics_yearly.csv'))
df_master_epi_demo.columns = df_master_epi_demo.columns.str.strip()

df_health_profile = pd.read_csv(os.path.join(data_dir, 'cleaned_health_demographics_final.csv'))
df_health_profile.columns = df_health_profile.columns.str.strip()

df_tech_ghs_raw = df_ict_ghs_cleaned.copy()
df_tech_ghs_raw.columns = df_tech_ghs_raw.columns.str.strip()

year_col_epi = 'YEAR' if 'YEAR' in df_master_epi_demo.columns else 'year'

# =====================================================================
# 2. סינון שנת 2021 (X) ושנת 2022 (Y) - מניעת זליגה וכפל שורות
# =====================================================================
print("2. מסנן שנים ומחשב את X4, X7 ואת משתנה המטרה Y...")
df_2021_epi = df_master_epi_demo[df_master_epi_demo[year_col_epi] == 2021].copy()

# חישוב X7 (אחוז בני +60)
total_pop_60_plus = (
    df_2021_epi.get('population_age_60_69', 0) +
    df_2021_epi.get('population_age_70_79', 0) +
    df_2021_epi.get('population_age_80_and_older', 0)
)
df_2021_epi['X7_calculated'] = (total_pop_60_plus / df_2021_epi['population']) * 100

# חישוב ה-Y משנת 2022
df_2022_epi = df_master_epi_demo[df_master_epi_demo[year_col_epi] == 2022].copy()
df_2022_epi['excess_mortality_rate'] = (df_2022_epi['new_deceased'] / df_2022_epi['population']) * 100000
df_2022_clean = df_2022_epi[df_2022_epi['excess_mortality_rate'] > 0].copy()

global_median = df_2022_clean['excess_mortality_rate'].median()
df_2022_clean['Y_is_ready'] = (df_2022_clean['excess_mortality_rate'] <= global_median).astype(int)
df_Y = df_2022_clean[['location_key', 'Y_is_ready']].copy()

# בניית טבלת הבסיס (הורדנו מכאן את population_density ו-human_development_index!)
df_base_x = df_2021_epi[['location_key', 'X4', 'X7_calculated']].copy()
df_master_final = pd.merge(df_base_x, df_Y, on='location_key', how='inner')

# =====================================================================
# 3. מיזוג נתוני בריאות (X1, X2)
# =====================================================================
print("3. מושך את נתוני המיטות (X1) והרופאים (X2)...")
beds_col = [col for col in df_health_profile.columns if 'bed' in col.lower()][0] if any('bed' in col.lower() for col in df_health_profile.columns) else None
physicians_col = [col for col in df_health_profile.columns if 'physician' in col.lower() or 'doctor' in col.lower()][0] if any('physician' in col.lower() or 'doctor' in col.lower() for col in df_health_profile.columns) else None

cols_to_pull_health = ['location_key']
rename_health_dict = {}
if beds_col:
    cols_to_pull_health.append(beds_col)
    rename_health_dict[beds_col] = 'X1_hospital_beds'
if physicians_col:
    cols_to_pull_health.append(physicians_col)
    rename_health_dict[physicians_col] = 'X2_physicians'

df_health_sub = df_health_profile[cols_to_pull_health].copy()
df_health_sub = df_health_sub.rename(columns=rename_health_dict)
df_master_final = pd.merge(df_master_final, df_health_sub, on='location_key', how='left')

# =====================================================================
# 4. מיזוג נתוני טכנולוגיה ו-GHS מהזיכרון (X3, X5, X6, X8, X9)
# =====================================================================
print("4. מושך את נתוני הטכנולוגיה וה-GHS כולל עישון וסוכרת...")

year_col_tech = 'Year' if 'Year' in df_tech_ghs_raw.columns else 'YEAR' if 'YEAR' in df_tech_ghs_raw.columns else None
df_tech_2021 = df_tech_ghs_raw[df_tech_ghs_raw[year_col_tech] == 2021].copy() if year_col_tech else df_tech_ghs_raw.drop_duplicates(subset=['location_key']).copy()

# איתור מדויק של עמודות סוכרת ועישון בכל רחבי הקובץ שלכן
diabetes_col = [col for col in df_tech_2021.columns if 'diabetes' in col.lower() or 'diab' in col.lower()]
smoking_col = [col for col in df_tech_2021.columns if 'smoking' in col.lower() or 'smok' in col.lower()]
ghs_early_col = [col for col in df_tech_2021.columns if 'early' in col.lower() or 'detection' in col.lower()]

cols_to_pull_tech = ['location_key', 'OVERALL SCORE', 'Internet users']
rename_tech_dict = {'OVERALL SCORE': 'X5_ghs_index', 'Internet users': 'X6_internet_users'}

if ghs_early_col:
    cols_to_pull_tech.append(ghs_early_col[0])
    rename_tech_dict[ghs_early_col[0]] = 'X3_ghs_early_detection'
if smoking_col:
    cols_to_pull_tech.append(smoking_col[0])
    rename_tech_dict[smoking_col[0]] = 'X8_smoking'
if diabetes_col:
    cols_to_pull_tech.append(diabetes_col[0])
    rename_tech_dict[diabetes_col[0]] = 'X9_diabetes'

df_tech_sub = df_tech_2021[[col for col in cols_to_pull_tech if col in df_tech_2021.columns]].copy()
df_tech_sub = df_tech_sub.rename(columns=rename_tech_dict)
df_master_final = pd.merge(df_master_final, df_tech_sub, on='location_key', how='left')

# =====================================================================
# 5. ארגון סדר העמודות הסופי (X1 עד X9 בלבד!) ומילוי חציונים
# =====================================================================
print("5. מנקה עמודות עודפות ומארגן את המטריצה מ-X1 עד X9...")

df_master_final = df_master_final.rename(columns={'X4': 'X4_testing_rate', 'X7_calculated': 'X7_age_60_plus'})

# וידוא קיום פיזי של עמודות X3, X8, X9 (במידה ולא אותרו בקובץ שלכן, הן ייווצרו כעת באופן אוטומטי)
if 'X3_ghs_early_detection' not in df_master_final.columns: df_master_final['X3_ghs_early_detection'] = np.nan
if 'X8_smoking' not in df_master_final.columns: df_master_final['X8_smoking'] = np.nan
if 'X9_diabetes' not in df_master_final.columns: df_master_final['X9_diabetes'] = np.nan

# השלמת חציונים אוטומטית לעמודות החסרות
numeric_cols = df_master_final.select_dtypes(include=['number']).columns
for col in numeric_cols:
    if col != 'Y_is_ready':
        df_master_final[col] = df_master_final[col].fillna(df_master_final[col].median())

# סידור הסדר הכרונולוגי הסופי - מוריד לחלוטין את population_density ו-HDI [1]
final_order = [
    'location_key',
    'X1_hospital_beds',
    'X2_physicians',
    'X3_ghs_early_detection',
    'X4_testing_rate',
    'X5_ghs_index',
    'X6_internet_users',
    'X7_age_60_plus',
    'X8_smoking',
    'X9_diabetes',
    'Y_is_ready'
]
df_master_final = df_master_final[[col for col in final_order if col in df_master_final.columns]]

# שמירה סופית
df_master_final.to_csv(os.path.join(data_dir, 'master_model_feature_matrix.csv'), index=False)

print(f"\n--- 🌟 המטריצה הרשמית והנקייה מוכנה! 🌟 ---")
print(f"📊 מימדי הטבלה הסופיים: {df_master_final.shape}")
display(df_master_final.head(15))


1. טוען את קבצי המקור המעודכנים...
2. מסנן שנים ומחשב את X4, X7 ואת משתנה המטרה Y...
3. מושך את נתוני המיטות (X1) והרופאים (X2)...
4. מושך את נתוני הטכנולוגיה וה-GHS כולל עישון וסוכרת...
5. מנקה עמודות עודפות ומארגן את המטריצה מ-X1 עד X9...

--- 🌟 המטריצה הרשמית והנקייה מוכנה! 🌟 ---
📊 מימדי הטבלה הסופיים: (219, 11)


,location_key,X1_hospital_beds,X2_physicians,X3_ghs_early_detection,X4_testing_rate,X5_ghs_index,X6_internet_users,X7_age_60_plus,X8_smoking,X9_diabetes,Y_is_ready
0,AD,2.3,3.33330,0.0,0.000000,34.70,93.89750,36.201385,NaN,NaN,0
1,AE,2.3,2.52780,66.7,9210.469850,39.60,100.00000,3.144635,NaN,NaN,1
2,AF,0.5,0.27820,33.3,0.000000,28.80,16.51430,4.222826,NaN,NaN,1
3,AG,2.3,2.95600,33.3,0.000000,36.65,75.44895,14.089944,NaN,NaN,0
4,AI,2.3,1.57655,33.3,0.000000,36.65,75.44895,15.284629,NaN,NaN,0
5,AL,2.3,1.21640,16.7,411.206993,45.00,79.32370,19.510960,NaN,NaN,1
6,AM,4.2,4.40230,66.7,667.976947,61.80,78.61230,18.453723,NaN,NaN,0
7,AO,2.3,0.21460,16.7,0.000000,29.10,39.38760,3.657909,NaN,NaN,1
8,AR,2.3,3.99010,66.7,380.272069,54.40,87.15070,12.741438,NaN,NaN,0
9,AS,2.3,1.57655,33.3,0.000000,36.65,75.44895,1036.665036,NaN,NaN,0


In [34]:
print(f"מספר המדינות הסופי במטריצה: {df_master_final.shape[0]}")


מספר המדינות הסופי במטריצה: 219
